In [ ]:
import torch

print(f"torch version: {torch.__version__}")
print(f"torch cuda version: {torch.version.cuda}")
device = torch.device(torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print(f"torch device: {device}")
print(f"device name: {torch.cuda.get_device_name(device) if torch.cuda.is_available() else 'cpu'}")

torch version: 2.6.0+cu126
torch cuda version: 12.6
torch device: cuda:0
device name: NVIDIA GeForce RTX 4090 Laptop GPU


In [ ]:
from common import MissionType, logger
from mistral_config import MistralConfig
from mistral_data import MistralData
from mistral_model import MistralModel


def run_task(mission_type: MissionType) -> None:
    Data = None
    Config = None
    match mission_type:
        case MissionType.WA:
            logger.info("Running WA task...")
            Data = MistralData(MissionType.WA)
            Config = MistralConfig(MissionType.WA)
        case MissionType.SA:
            logger.info("Running SA task...")
            Data = MistralData(MissionType.SA)
            Config = MistralConfig(MissionType.SA)
        case MissionType.SE:
            logger.info("Running SE task...")
            Data = MistralData(MissionType.SE)
            Config = MistralConfig(MissionType.SE)
        case _:
            raise ValueError(f"Invalid mission type: {mission_type}")
    Model = MistralModel(Config, Data)

    logger.info("Training the model...")

    Model.train()

    logger.info("Model training completed.")

    Model.save()

    logger.info("Model saved successfully.")

    Model.predict(Data)

    logger.info("Model prediction completed.")

    Model.report(Data)

    logger.info("Model report completed.")
    logger.info("All tasks completed successfully.")

In [ ]:
run_task(MissionType.WA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
)


se_merge_model = AutoModelForCausalLM.from_pretrained(
    "./mistral-7b-lora-SE",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

se_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
se_tokenizer.pad_token = se_tokenizer.eos_token

In [ ]:
FuncFindAnswer = lambda text: re.search(r"Answer: (.*)", text).group(1).strip() if re.search(r"Answer: (.*)", text) else "None"

def predict(Model, Tokenizer, MistralData: MistralData) -> list:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info("Predicting...")
    # for i in tqdm.tqdm(range(len(test_data))):
    for i in range(92, 97):
        input_text = test_data["input"][i]
        print(f"Input: {input_text[:100]}")
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = Tokenizer(format_input, return_tensors="pt").input_ids.to(
            device
        )
        attention_mask = Tokenizer(
            format_input, return_tensors="pt"
        ).attention_mask.to(device)
        Model.gradient_checkpointing_enable()
        Model.generation_config.pad_token_id = Tokenizer.pad_token_id
        output = Model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1,
        )
        Model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = Tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = FuncFindAnswer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred

y_pred = predict(se_merge_model, se_tokenizer, MistralData(MissionType.SE))
y_test = MistralData(MissionType.SE).df_test["y_true"].tolist()

print(f"y_test: {y_test[92:97]}")
print(f"y_pred: {y_pred}")

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: \t {accuracy:.8f}")
# Calculate precision

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Precision: \t {precision:.8f}")

# Calculate recall
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Recall: \t {recall:.8f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"F1 Score: \t {f1:.8f}")

# Generate classification report
report = classification_report(y_test, y_pred, zero_division=0)
print(f"Classification Report: \n{report}")

Accuracy: 	 0.70682148
Precision: 	 0.70394435
Recall: 	 0.70682148
F1 Score: 	 0.70128947
Classification Report: 
                             precision    recall  f1-score   support

        1st year apprentice       0.00      0.00      0.00         1
    Chief Executive Officer       0.00      0.00      0.00         0
                   advanced       1.00      1.00      1.00         1
                 apprentice       0.75      1.00      0.86         3
                  assistant       0.77      0.86      0.81        28
        assistant principal       0.00      0.00      0.00         0
                  associate       0.83      0.83      0.83         6
         associate director       0.67      1.00      0.80         2
         associate-director       0.00      0.00      0.00         1
                      cadet       1.00      0.50      0.67         2
                      chief       1.00      0.67      0.80         3
                 consultant       0.00      0.00      0.

In [ ]:
FuncFindAnswer = lambda text: re.search(r"Answer: (.*)", text).group(1).strip() if re.search(r"Answer: (.*)", text) else "None"

def predict(Model, Tokenizer, MistralData: MistralData) -> list:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info("Predicting...")
    for i in tqdm.tqdm(range(len(test_data))):
        input_text = test_data["input"][i]
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = Tokenizer(format_input, return_tensors="pt").input_ids.to(
            device
        )
        attention_mask = Tokenizer(
            format_input, return_tensors="pt"
        ).attention_mask.to(device)
        Model.gradient_checkpointing_enable()
        Model.generation_config.pad_token_id = Tokenizer.pad_token_id
        output = Model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1,
        )
        Model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = Tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = FuncFindAnswer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred

y_pred = predict(sa_merge_model, sa_tokenizer, MistralData(MissionType.SA))
y_test = MistralData(MissionType.SA).df_test["y_true"].tolist()

print(f"y_test: {y_test}")
print(f"y_pred: {y_pred}")

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: \t {accuracy:.8f}")
# Calculate precision

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Precision: \t {precision:.8f}")

# Calculate recall
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Recall: \t {recall:.8f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"F1 Score: \t {f1:.8f}")

# Generate classification report
report = classification_report(y_test, y_pred, zero_division=0)
print(f"Classification Report: \n{report}")

Accuracy: 	 0.85361552
Precision: 	 0.84672669
Recall: 	 0.85361552
F1 Score: 	 0.84766150
Classification Report: 
                           precision    recall  f1-score   support

            0-0-None-None       0.97      0.99      0.98       238
         10-10-NZD-HOURLY       0.00      0.00      0.00         0
         10-10-SGD-HOURLY       1.00      1.00      1.00         1
         10-40-NZD-HOURLY       0.00      0.00      0.00         1
       100-100-HKD-HOURLY       1.00      0.67      0.80         3
       100-120-HKD-HOURLY       0.67      1.00      0.80         2
       100-250-THB-HOURLY       1.00      1.00      1.00         2
     1000-1000-THB-HOURLY       1.00      1.00      1.00         1
       100000-100000-THB-       0.00      0.00      0.00         0
100000-100000-THB-MONTHLY       0.00      0.00      0.00         1
       100000-130000-AUD-       0.00      0.00      0.00         0
       102051-125151-AUD-       0.00      0.00      0.00         0
 102051-12515

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
)


wa_merge_model = AutoModelForCausalLM.from_pretrained(
    "./mistral-7b-lora-WA",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

wa_tokenizer = AutoTokenizer.from_pretrained("./mistral-7b-lora-WA")
wa_tokenizer.pad_token = wa_tokenizer.eos_token

2025-04-11 13:35:54 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
FuncFindAnswer = lambda text: re.search(r"Answer: (.*)", text).group(1).strip() if re.search(r"Answer: (.*)", text) else "None"

def predict(Model, Tokenizer, MistralData: MistralData) -> list:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info(f"Predicting... {len(test_data)}")
    for i in tqdm.tqdm(range(len(test_data))):
        input_text = test_data["input"][i]
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = Tokenizer(format_input, return_tensors="pt").input_ids.to(
            device
        )
        attention_mask = Tokenizer(
            format_input, return_tensors="pt"
        ).attention_mask.to(device)
        Model.gradient_checkpointing_enable()
        Model.generation_config.pad_token_id = Tokenizer.pad_token_id
        output = Model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1,
        )
        Model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = Tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = FuncFindAnswer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred

y_pred = predict(wa_merge_model, wa_tokenizer, MistralData(MissionType.WA))
y_test = MistralData(MissionType.WA).df_test["y_true"].tolist()

print(f"y_test: {y_test}")
print(f"y_pred: {y_pred}")

2025-04-11 13:46:03 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 39433.63it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-11 13:46:04 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 47766.72it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-11 13:46:04 - INFO - Predicting... 99
100%|██████████| 99/99 [00:54<00:00,  1.83it/s]
2025-04-11 13:46:58 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 49385.83it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-11 13:46:59 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 49533.11it/s]


Generating train split: 0 examples [00:00, ? examples/s]

y_test: ['OnSite', 'OnSite', 'Remote', 'Hybrid', 'Remote', 'OnSite', 'Hybrid', 'Remote', 'Hybrid', 'Remote', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'Remote', 'Hybrid', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'Hybrid', 'Hybrid', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'Hybrid', 'Hybrid', 'OnSite', 'Hybrid', 'Remote', 'Remote', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'Remote', 'Remote', 'OnSite', 'Hybrid', 'Hybrid', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Hybrid', 'Hybrid', 'Remote', 'OnSite', 'Hybrid', 'OnSite', 'Hybrid', 'Remote', 'Remote', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'Hybrid', 'OnSite', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'OnSite', 'Hybrid', 'Remote', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Hybrid', 'OnSite', 'OnSite', 'OnSite']
y

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: \t {accuracy:.8f}")
# Calculate precision

precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"Precision: \t {precision:.8f}")

# Calculate recall
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Recall: \t {recall:.8f}")

# Calculate F1 score
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
print(f"F1 Score: \t {f1:.8f}")

# Generate classification report
report = classification_report(y_test, y_pred, zero_division=0)
print(f"Classification Report: \n{report}")

Accuracy: 	 0.84848485
Precision: 	 0.84848485
Recall: 	 0.84848485
F1 Score: 	 0.84848485
Classification Report: 
              precision    recall  f1-score   support

      Hybrid       0.74      0.74      0.74        27
      OnSite       0.89      0.89      0.89        46
      Remote       0.88      0.88      0.88        26

    accuracy                           0.85        99
   macro avg       0.84      0.84      0.84        99
weighted avg       0.85      0.85      0.85        99

